# Regression NBA Model


## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training,
)
from nba_ou.modeling.modeling import (
    TemporalDecaySampleWeightRegressor,
    evaluate_day_by_day_walk_forward,
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
    save_model_bundle,
    load_model_bundle,
)


In [2]:
TARGET_COL = "LINE_ERROR"
SAMPLE_WEIGHT_LAMBDA = 0.005
SAMPLE_WEIGHT_LAMBDA_BOUNDS = (1e-4, 0.01)
TRAIN_GAMES = 5000

## Load Data

In [3]:
nan_threshold = 50.0
max_na_per_row = 80


data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260405.csv"

path = data_path + name

header_cols = pd.read_csv(path, nrows=0).columns
dtype_dict = {col: str for col in header_cols if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict,
)
df_stats["GAME_DATE"] = pd.to_datetime(df_stats["GAME_DATE"]).dt.strftime("%Y-%m-%d")
df_stats = df_stats[df_stats['SEASON_YEAR'] >= 2021].copy()


In [4]:
import time
time.sleep(20*60)

In [5]:
exclude = "fanatics_sportsbook"

In [6]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'], exclude_cols_containing=[exclude])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 6456 rows
Basic cleaning complete: 6441 rows remaining

Starting advanced column cleaning with 2948 columns

Advanced column cleaning complete: 2948 → 2053 columns (895 removed)


Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 1 (0.02%)
  Critical columns requiring data: 4
  Columns zero-filled: 112
  Infer pairs applied: 0/106
  Remaining NaN cells: 251127

Dropping rows with more than 80 NaN values...
Removed 618 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (5822, 2053)


In [7]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and "SEASON_YEAR" in df_to_train.columns:
            common_season = null_rows["SEASON_YEAR"].mode()
            most_common_season.append(
                common_season.iloc[0] if len(common_season) > 0 else None
            )
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame(
    {
        "Column": na_counts.index,
        "NA_Count": na_counts.values,
        "NA_Percentage": (na_counts.values / len(df_to_train) * 100).round(2),
        "Most_Common_Season_Year": most_common_season,
    }
).sort_values("NA_Count", ascending=False)

na_counts_df[na_counts_df["NA_Count"] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1731,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,792,13.60,2023.0
1729,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,778,13.36,2023.0
1735,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,709,12.18,2023.0
1733,spread_consensus_pct_away_TREND_SLOPE_LAST_5_H...,702,12.06,2023.0
1730,total_consensus_pct_under_TREND_SLOPE_LAST_5_G...,684,11.75,2023.0
...,...,...,...,...
1917,odds_book_total_prob_diff_novig_betmgm,1,0.02,2021.0
1874,total_betmgm_price_under,1,0.02,2021.0
1885,spread_betmgm_price_home,1,0.02,2021.0
1337,DIFF_FROM_LINE_caesars_LAST_ALL_2_MATCHES_DIFF...,1,0.02,2023.0


In [8]:
BET365_LINE_COL = "TOTAL_LINE_bet365"
# BET365_LINE_COL = "total_bet365_line_over"

# Ensure the main scoring line and actual total exist.
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [9]:
df_to_train["LINE_ERROR"] = df_to_train["TOTAL_POINTS"] - df_to_train[BET365_LINE_COL]


In [10]:
df_to_train["GAME_DATE"] = pd.to_datetime(df_to_train["GAME_DATE"])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

# Count games per season
games_per_season = df_to_train.groupby("SEASON_YEAR").size()
print(games_per_season)


SEASON_YEAR
2021    1239
2022    1235
2023    1010
2024    1238
2025    1100
dtype: int64


## Train / Test

In [11]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
)
from sklearn.model_selection import cross_validate
from xgboost import XGBRegressor

from nba_ou.modeling.optuna_error_line import (
    fit_best_xgb_error_line,
    select_best_trial_lexicographic,
    summarize_lexicographic_candidates,
    summarize_optuna_trials,
    tune_xgb_error_line_optuna,
)
from nba_ou.modeling.scorers import (
    OverUnderScorerLineError,
    OverUnderScorerLineErrorMinEdge,
    evaluate_error_thresholds,
    over_under_betting_accuracy_error_line,
    over_under_betting_accuracy_error_line_with_min_edge,
)

In [12]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.08,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print(
    "Final test date range:",
    df_test_final["GAME_DATE"].min(),
    "->",
    df_test_final["GAME_DATE"].max(),
)

Development set size: 5348
Final test set size: 474
Final test date range: 2026-01-28 00:00:00 -> 2026-04-05 00:00:00


In [13]:
def build_recency_sample_weights(df, date_col="GAME_DATE", lambda_=SAMPLE_WEIGHT_LAMBDA):
    dates = pd.to_datetime(df[date_col])
    max_date = dates.max()
    age_days = (max_date - dates).dt.days
    weights = np.exp(-lambda_ * age_days)
    return pd.Series(weights, index=df.index, name="sample_weight")

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "LINE_ERROR",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")
sample_weight_dev = build_recency_sample_weights(df_dev)

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")
print(
    f"Recency sample weights lambda={SAMPLE_WEIGHT_LAMBDA}: "
    f"min={sample_weight_dev.min():.4f}, max={sample_weight_dev.max():.4f}"
)


X_dev shape: (5348, 2050)
X_test_final shape: (474, 2050)
Recency sample weights lambda=0.005: min=0.0004, max=1.0000


In [14]:
ou_scorer = OverUnderScorerLineError()
ou_scorer_edge_2 = OverUnderScorerLineErrorMinEdge(min_edge=2)
ou_scorer_edge_4 = OverUnderScorerLineErrorMinEdge(min_edge=4)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
    "OU_Betting_Accuracy_Edge_2": ou_scorer_edge_2,
    "OU_Betting_Accuracy_Edge_4": ou_scorer_edge_4,
}


def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc.startswith("OU_Betting_Accuracy"):
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()


In [15]:
DAY_BY_DAY_METRIC_NAME = "OU_Betting_Accuracy"
DAY_BY_DAY_THRESHOLDS = (1, 2, 3)


def summarize_walk_forward_thresholds(predictions_df, thresholds=DAY_BY_DAY_THRESHOLDS):
    y_true = pd.to_numeric(predictions_df["y_true"], errors="coerce").to_numpy(dtype=float)
    y_pred = pd.to_numeric(predictions_df["y_pred"], errors="coerce").to_numpy(dtype=float)
    margin = np.abs(y_pred)
    n_total = len(predictions_df)

    rows = []
    for t in thresholds:
        mask = margin > t
        n = int(mask.sum())
        acc = (
            np.nan
            if n == 0
            else over_under_betting_accuracy_error_line(
                y_true_error=y_true[mask],
                y_pred_error=y_pred[mask],
            )
        )
        rows.append(
            {
                "threshold_abs_pred_error_gt": t,
                "n_games": n,
                "pct_of_test": (n / n_total) if n_total else np.nan,
                "directional_accuracy": acc,
            }
        )

    return pd.DataFrame(rows)


def run_day_by_day_walk_forward_evaluation(
    *,
    label,
    df_dev,
    df_test_final,
    fit_and_predict,
    max_games=TRAIN_GAMES,
    metric_name=DAY_BY_DAY_METRIC_NAME,
    thresholds=DAY_BY_DAY_THRESHOLDS,
):
    result = evaluate_day_by_day_walk_forward(
        df_dev=df_dev,
        df_test_final=df_test_final,
        fit_and_predict=fit_and_predict,
        metric_fn=lambda y_true, y_pred: over_under_betting_accuracy_error_line(
            y_true_error=y_true,
            y_pred_error=y_pred,
        ),
        target_col=TARGET_COL,
        max_games=max_games,
        metric_name=metric_name,
    )

    threshold_results = summarize_walk_forward_thresholds(
        result.predictions,
        thresholds=thresholds,
    )

    print(f"{label} mean day-by-day {metric_name}: {result.mean_metric:.2%}")
    display(result.daily_results.style.format({metric_name: "{:.2%}"}))
    print(f"{label} thresholded walk-forward accuracy")
    display(
        threshold_results.style.format(
            {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
        )
    )
    return result, threshold_results


In [16]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=25,
    step_games_between_tests=50,
    train_games=TRAIN_GAMES,
    min_train_games=TRAIN_GAMES*0.75,
    max_folds=15,
    verbose=1,
)

assert_valid_time_splits(df_dev, splits)



Created 15 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           4085            28       2021-10-25     2025-01-25      2025-01-26    2025-01-29         2024
    2           4167            27       2021-10-25     2025-02-05      2025-02-06    2025-02-09         2024
    3           4247            30       2021-10-25     2025-02-21      2025-02-22    2025-02-25         2024
    4           4332            30       2021-10-25     2025-03-04      2025-03-05    2025-03-08         2024
    5           4418            33       2021-10-25     2025-03-15      2025-03-16    2025-03-19         2024
    6           4501            30       2021-10-25     2025-03-26      2025-03-27    2025-03-30         2024
    7           4587            27       2021-10-25     2025-04-06      2025-04-07    2025-04-10         2024
    8           4722            32       2021-10-25     2025-06-22      2025

In [17]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)


DummyRegressor baseline
Train MSE: 288.45885
Validation MSE: 308.74456

Train RMSE: 16.98405
Validation RMSE: 17.44733

Train MAE: 13.51229
Validation MAE: 13.78484

Train R2: 0.00000
Validation R2: -0.04183

Train OU_Betting_Accuracy: 51.64%
Validation OU_Betting_Accuracy: 52.54%

Train OU_Betting_Accuracy_Edge_2: 0.00%
Validation OU_Betting_Accuracy_Edge_2: 0.00%

Train OU_Betting_Accuracy_Edge_4: 0.00%
Validation OU_Betting_Accuracy_Edge_4: 0.00%



In [18]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)


Linear Regression
Train MSE: 187.87288
Validation MSE: 1165464.10092

Train RMSE: 13.70487
Validation RMSE: 300.91618

Train MAE: 10.78246
Validation MAE: 71.69889

Train R2: 0.34864
Validation R2: -2642.20472

Train OU_Betting_Accuracy: 69.94%
Validation OU_Betting_Accuracy: 49.50%

Train OU_Betting_Accuracy_Edge_2: 73.50%
Validation OU_Betting_Accuracy_Edge_2: 50.59%

Train OU_Betting_Accuracy_Edge_4: 77.14%
Validation OU_Betting_Accuracy_Edge_4: 50.27%



In [19]:
xgb_reg_no_weights = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=35,
    subsample=0.65,
    colsample_bytree=0.68,
    reg_alpha=5.28,
    reg_lambda=1.3,
    min_child_weight=5.08,
    gamma=0.0085,
    n_jobs=-1,
    random_state=16,
)

cv_results_no_weights = cross_validate(
    xgb_reg_no_weights,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost no sample weights")
print_metrics(cv_results_no_weights)

XGBoost no sample weights
Train MSE: 266.22867
Validation MSE: 307.83147

Train RMSE: 16.31644
Validation RMSE: 17.41995

Train MAE: 12.96968
Validation MAE: 13.70912

Train R2: 0.07704
Validation R2: -0.03851

Train OU_Betting_Accuracy: 63.61%
Validation OU_Betting_Accuracy: 56.69%

Train OU_Betting_Accuracy_Edge_2: 86.35%
Validation OU_Betting_Accuracy_Edge_2: 51.07%

Train OU_Betting_Accuracy_Edge_4: 98.17%
Validation OU_Betting_Accuracy_Edge_4: 6.67%



In [20]:
xgb_reg_no_weights.fit(X_dev, y_dev)

y_pred_test_error = xgb_reg_no_weights.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 303.52747
RMSE: 17.42204
MAE: 13.77651
OU_Betting_Accuracy: 49.46%
OU_Betting_Accuracy_Edge_2: 51.52%
OU_Betting_Accuracy_Edge_4: nan%


In [21]:
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=xgb_reg_no_weights,
    X_test=X_test_final,
    y_test_error=y_test_final,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,474,100.0%,49.46%
1,1,167,35.2%,49.69%
2,2,33,7.0%,51.52%
3,3,3,0.6%,33.33%
4,4,0,0.0%,nan%
5,5,0,0.0%,nan%
6,6,0,0.0%,nan%
7,7,0,0.0%,nan%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


In [22]:
def fit_and_predict_xgb_no_weights_day_by_day(train_df, test_df):
    model = XGBRegressor(**xgb_reg_no_weights.get_params())

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_no_weights, day_by_day_no_weights_thresholds = run_day_by_day_walk_forward_evaluation(
    label="XGBoost no sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_no_weights_day_by_day,
    max_games=TRAIN_GAMES,
)

XGBoost no sample weights mean day-by-day OU_Betting_Accuracy: 50.53%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-01-28 00:00:00,5000,9,2021-12-15 00:00:00,2026-01-27 00:00:00,77.78%
1,2026-01-29 00:00:00,5000,8,2021-12-16 00:00:00,2026-01-28 00:00:00,62.50%
2,2026-01-30 00:00:00,5000,9,2021-12-17 00:00:00,2026-01-29 00:00:00,25.00%
3,2026-01-31 00:00:00,5000,6,2021-12-19 00:00:00,2026-01-30 00:00:00,50.00%
4,2026-02-01 00:00:00,5000,10,2021-12-20 00:00:00,2026-01-31 00:00:00,60.00%
5,2026-02-02 00:00:00,5000,4,2021-12-22 00:00:00,2026-02-01 00:00:00,25.00%
6,2026-02-03 00:00:00,5000,10,2021-12-22 00:00:00,2026-02-02 00:00:00,60.00%
7,2026-02-04 00:00:00,5000,7,2021-12-23 00:00:00,2026-02-03 00:00:00,42.86%
8,2026-02-05 00:00:00,5000,8,2021-12-26 00:00:00,2026-02-04 00:00:00,50.00%
9,2026-02-06 00:00:00,5000,6,2021-12-27 00:00:00,2026-02-05 00:00:00,66.67%


XGBoost no sample weights thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,209,44.1%,51.71%
1,2,64,13.5%,48.44%
2,3,7,1.5%,42.86%


## Check weighted

In [23]:
xgb_reg_weights = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=35,
    subsample=0.65,
    colsample_bytree=0.68,
    reg_alpha=5.28,
    reg_lambda=1.3,
    min_child_weight=5.08,
    gamma=0.0085,
    n_jobs=-1,
    random_state=16,
)

weighted_xgb = TemporalDecaySampleWeightRegressor(
    estimator=xgb_reg_weights,
    dates=df_dev["GAME_DATE"],
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

cv_results_weights = cross_validate(
    weighted_xgb,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost with sample weights (per-fold decay)")
print_metrics(cv_results_weights)

XGBoost with sample weights (per-fold decay)
Train MSE: 274.66546
Validation MSE: 311.16197

Train RMSE: 16.57290
Validation RMSE: 17.52071

Train MAE: 13.16063
Validation MAE: 13.79354

Train R2: 0.04780
Validation R2: -0.05148

Train OU_Betting_Accuracy: 58.30%
Validation OU_Betting_Accuracy: 52.23%

Train OU_Betting_Accuracy_Edge_2: 68.25%
Validation OU_Betting_Accuracy_Edge_2: 55.80%

Train OU_Betting_Accuracy_Edge_4: 86.19%
Validation OU_Betting_Accuracy_Edge_4: 34.00%



In [24]:
weighted_xgb.fit(X_dev, y_dev)

y_pred_test_error = weighted_xgb.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 302.58264
RMSE: 17.39490
MAE: 13.75500
OU_Betting_Accuracy: 52.26%
OU_Betting_Accuracy_Edge_2: 52.32%
OU_Betting_Accuracy_Edge_4: 56.67%


In [25]:
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=weighted_xgb,
    X_test=X_test_final,
    y_test_error=y_test_final,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,474,100.0%,52.26%
1,1,349,73.6%,52.92%
2,2,243,51.3%,52.32%
3,3,137,28.9%,55.64%
4,4,64,13.5%,56.67%
5,5,19,4.0%,57.89%
6,6,3,0.6%,33.33%
7,7,0,0.0%,nan%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


In [26]:
def fit_and_predict_xgb_weights_day_by_day(train_df, test_df):
    base_model = XGBRegressor(**xgb_reg_weights.get_params())
    model = TemporalDecaySampleWeightRegressor(
        estimator=base_model,
        dates=train_df["GAME_DATE"],
        lambda_=SAMPLE_WEIGHT_LAMBDA,
    )

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_weights, day_by_day_weights_thresholds = run_day_by_day_walk_forward_evaluation(
    label="XGBoost with sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_weights_day_by_day,
    max_games=TRAIN_GAMES,
)


XGBoost with sample weights mean day-by-day OU_Betting_Accuracy: 53.14%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-01-28 00:00:00,5000,9,2021-12-15 00:00:00,2026-01-27 00:00:00,77.78%
1,2026-01-29 00:00:00,5000,8,2021-12-16 00:00:00,2026-01-28 00:00:00,62.50%
2,2026-01-30 00:00:00,5000,9,2021-12-17 00:00:00,2026-01-29 00:00:00,37.50%
3,2026-01-31 00:00:00,5000,6,2021-12-19 00:00:00,2026-01-30 00:00:00,33.33%
4,2026-02-01 00:00:00,5000,10,2021-12-20 00:00:00,2026-01-31 00:00:00,80.00%
5,2026-02-02 00:00:00,5000,4,2021-12-22 00:00:00,2026-02-01 00:00:00,50.00%
6,2026-02-03 00:00:00,5000,10,2021-12-22 00:00:00,2026-02-02 00:00:00,50.00%
7,2026-02-04 00:00:00,5000,7,2021-12-23 00:00:00,2026-02-03 00:00:00,57.14%
8,2026-02-05 00:00:00,5000,8,2021-12-26 00:00:00,2026-02-04 00:00:00,50.00%
9,2026-02-06 00:00:00,5000,6,2021-12-27 00:00:00,2026-02-05 00:00:00,83.33%


XGBoost with sample weights thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,301,63.5%,56.23%
1,2,177,37.3%,53.76%
2,3,91,19.2%,53.33%


# Optuna

In [27]:
study = tune_xgb_error_line_optuna(
    X=X_dev,
    y=y_dev,
    sample_weight_dates=df_dev["GAME_DATE"],
    tune_sample_weight_lambda=True,
    sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    splits=splits,
    n_trials=80,
    timeout=4.5 * 3600,
    # timeout=600,

    objective_name="reg:squarederror",
    study_name="xgb_error_line_mae",
)

best_trial_lexi = select_best_trial_lexicographic(
    study,
    mae_tolerance_abs=0.05,
)

print("Optuna best by MAE only")
print("Trial:", study.best_trial.number)
print("Best CV MAE:", study.best_value)
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", study.best_trial.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", study.best_trial.user_attrs.get("mean_ou_acc_edge_4"))

print("\nSelected trial after MAE-first / OU-second ranking")
print("Trial:", best_trial_lexi.number)
print("CV MAE:", best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value))
print("Mean RMSE:", best_trial_lexi.user_attrs.get("mean_rmse"))
print("Mean R2:", best_trial_lexi.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", best_trial_lexi.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_4"))
print("Median best_iteration:", best_trial_lexi.user_attrs.get("median_best_iteration"))
print("Params:")
for k, v in best_trial_lexi.params.items():
    print(f"{k}: {v}")

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)

candidates_df = summarize_lexicographic_candidates(
    study,
    mae_tolerance_abs=0.05,
)
display(
    candidates_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)


[I 2026-04-07 00:12:34,409] A new study created in memory with name: xgb_error_line_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-04-07 00:20:04,990] Trial 0 finished with value: 13.610897245641977 and parameters: {'max_depth': 2, 'min_child_weight': 18.346704707583235, 'gamma': 1.6970342240854253, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5123279759064977, 'learning_rate': 0.011926786034588454, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307, 'sample_weight_lambda': 0.00013824509574177668}. Best is trial 0 with value: 13.610897245641977.
[I 2026-04-07 00:32:07,730] Trial 1 finished with value: 13.55557260141377 and parameters: {'max_depth': 4, 'min_child_weight': 20.290108931287893, 'gamma': 0.3261777842309625, 'subsample': 0.8390562044499359, 'colsample_bytree': 0.4213034780854249, 'learning_rate': 0.012620826760486504, 'reg_alpha': 0.09307011182812809, 'reg_lambda': 15.258811505244246, 'sample_weight_lambda': 0.000848258413826022}. Best is trial 1 with value: 13.55557260141377.
[I 2026-04-07 00:36:37,568] Trial 2 finished with value: 13.617799053021935 and parameters: {'ma

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,12,13.3482,17.1736,-0.0102,57.32%,47.73%,0.463413,50.60%,136,131,3,51.448313,2.266313,0.843540,0.798779,0.052350,0.186194,47.613477,0.000316
1,6,13.4066,17.2642,-0.0218,57.48%,50.83%,0.354365,35.87%,122,34,4,21.354130,1.318300,0.833701,0.787420,0.040420,0.264001,42.379142,0.000125
2,11,13.4276,17.2267,-0.0169,58.04%,55.64%,0.452273,37.94%,117,87,3,40.900456,2.034268,0.829039,0.788311,0.053950,0.021864,40.152574,0.000438
3,25,13.4341,17.2762,-0.0221,59.01%,54.98%,0.402694,38.44%,127,95,2,38.214250,2.616392,0.730738,0.491414,0.058323,0.062371,12.517621,0.000270
4,31,13.4360,17.1854,-0.0116,59.12%,49.16%,0.378326,29.44%,99,51,2,36.146615,2.661068,0.737227,0.478582,0.058804,0.064826,13.434406,0.000254
5,32,13.4419,17.2070,-0.0144,56.88%,53.53%,0.320370,25.56%,107,40,2,19.607945,2.553958,0.662098,0.500889,0.054826,0.107698,30.448615,0.000148
6,14,13.4479,17.2743,-0.0230,59.00%,48.97%,0.368175,41.11%,56,49,4,8.859991,2.861623,0.764757,0.538307,0.058789,0.288475,5.490448,0.000115
7,16,13.4510,17.1958,-0.0126,57.35%,58.37%,0.399588,22.22%,86,40,3,39.314881,2.444286,0.704882,0.736309,0.024461,0.338286,3.238246,0.009988
8,17,13.4524,17.2966,-0.0242,59.83%,55.90%,0.522698,42.38%,87,78,4,10.261234,1.355467,0.818052,0.798621,0.041334,0.829286,25.155576,0.000181
9,30,13.4536,17.2206,-0.0167,57.68%,42.29%,0.341818,30.81%,65,38,4,41.460172,1.855081,0.849750,0.598873,0.052950,0.048881,39.043962,0.000353


,trial,value_mae,mean_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,mae_cutoff,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,12,13.3482,13.3482,17.1736,-0.0102,57.32%,47.73%,0.463413,50.60%,136,131,13.398198,3,51.448313,2.266313,0.843540,0.798779,0.052350,0.186194,47.613477,0.000316


In [28]:
def fit_and_predict_optuna_day_by_day(train_df, test_df):
    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model = fit_best_xgb_error_line(
        X_dev=X_train,
        y_dev=y_train,
        sample_weight_dates=train_df["GAME_DATE"],
        sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
        trial=best_trial_lexi,
        objective_name="reg:squarederror",
    )
    return model.predict(X_test)

day_by_day_optuna, day_by_day_optuna_thresholds = run_day_by_day_walk_forward_evaluation(
    label="Optuna-selected XGBoost",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_optuna_day_by_day,
    max_games=TRAIN_GAMES,
)

Optuna-selected XGBoost mean day-by-day OU_Betting_Accuracy: 54.25%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-01-28 00:00:00,5000,9,2021-12-15 00:00:00,2026-01-27 00:00:00,77.78%
1,2026-01-29 00:00:00,5000,8,2021-12-16 00:00:00,2026-01-28 00:00:00,62.50%
2,2026-01-30 00:00:00,5000,9,2021-12-17 00:00:00,2026-01-29 00:00:00,62.50%
3,2026-01-31 00:00:00,5000,6,2021-12-19 00:00:00,2026-01-30 00:00:00,66.67%
4,2026-02-01 00:00:00,5000,10,2021-12-20 00:00:00,2026-01-31 00:00:00,50.00%
5,2026-02-02 00:00:00,5000,4,2021-12-22 00:00:00,2026-02-01 00:00:00,50.00%
6,2026-02-03 00:00:00,5000,10,2021-12-22 00:00:00,2026-02-02 00:00:00,60.00%
7,2026-02-04 00:00:00,5000,7,2021-12-23 00:00:00,2026-02-03 00:00:00,42.86%
8,2026-02-05 00:00:00,5000,8,2021-12-26 00:00:00,2026-02-04 00:00:00,75.00%
9,2026-02-06 00:00:00,5000,6,2021-12-27 00:00:00,2026-02-05 00:00:00,66.67%


Optuna-selected XGBoost thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,301,63.5%,54.27%
1,2,171,36.1%,54.22%
2,3,98,20.7%,58.95%


In [29]:
total_df = df_dev.tail(TRAIN_GAMES)

In [30]:
X_dev = total_df.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(total_df[TARGET_COL], errors="coerce")
sample_weight_dates_dev = total_df["GAME_DATE"]


In [31]:
best_model = fit_best_xgb_error_line(
    X_dev=X_dev,
    y_dev=y_dev,
    sample_weight_dates=sample_weight_dates_dev,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

y_pred_test_error = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 304.59468
RMSE: 17.45264
MAE: 13.79930
OU_Betting_Accuracy: 52.26%
OU_Betting_Accuracy_Edge_2: 51.22%
OU_Betting_Accuracy_Edge_4: 65.22%


In [32]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

df_to_train_split_rows = df_to_train.copy()
df_to_train_split_rows = df_to_train_split_rows.tail(TRAIN_GAMES)

X_full = df_to_train_split_rows.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(df_to_train_split_rows[TARGET_COL], errors="coerce")
sample_weight_dates_full = df_to_train_split_rows["GAME_DATE"]

production_model = fit_best_xgb_error_line(
    X_dev=X_full,
    y_dev=y_full,
    sample_weight_dates=sample_weight_dates_full,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(df_to_train_split_rows["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"five_seasons_xgb_line_error_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="five_seasons_line_error",
        prediction_source="five_seasons_xgb_line_error",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=best_trial_lexi.params,
        selected_trial_number=best_trial_lexi.number,
        mean_best_iteration=best_trial_lexi.user_attrs.get("mean_best_iteration"),
        median_best_iteration=best_trial_lexi.user_attrs.get("median_best_iteration"),
        cv_mae=float(best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value)),
        cv_rmse=best_trial_lexi.user_attrs.get("mean_rmse"),
        cv_ou_acc=best_trial_lexi.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=df_to_train_split_rows["GAME_DATE"].min().to_pydatetime(),
        train_date_max=df_to_train_split_rows["GAME_DATE"].max().to_pydatetime(),
        train_games= TRAIN_GAMES,
        sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/5_seasons/",
    metadata=metadata,
)

print(
    f"Production model trained on {len(X_full)} rows using fixed n_estimators from median_best_iteration."
)
print("Saved model :", model_path)
print("Saved metadata:", meta_path)

Production model trained on 5000 rows using fixed n_estimators from median_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/5_seasons/five_seasons_xgb_line_error_05_04_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/5_seasons/five_seasons_xgb_line_error_05_04_26.meta.json


In [33]:
best_trial_lexi.user_attrs.get("median_best_iteration")


131